# 8. Persiapan Deployment Web
Notebook ini merangkum dan membersihkan fungsi prediksi agar siap diekspor ke dalam *backend* aplikasi Web (misalnya Flask, FastAPI, atau Streamlit). Kode divisualisasi (Matplotlib) dihilangkan agar fungsi mengembalikan nilai murni (*return values*).

In [1]:
import json

# Menyimpan class mapping ke JSON agar web tidak hardcode nama kelas
class_mapping = {
    "0": "COVID",
    "1": "Normal",
    "2": "Viral_Pneumonia"
}

with open('../models/class_indices.json', 'w') as f:
    json.dump(class_mapping, f)

print("Kamus kelas berhasil disimpan sebagai JSON di folder models!")

Kamus kelas berhasil disimpan sebagai JSON di folder models!


In [2]:
import numpy as np
from PIL import Image, UnidentifiedImageError
from tensorflow.keras.models import load_model

# --- MODULARISASI 1: Fungsi Khusus Preprocessing ---
def preprocess_image(img):
    """
    Pemrosesan gambar dengan standar keamanan dan optimasi Keras.
    Menggunakan enum Resampling terbaru agar tidak muncul DeprecationWarning.
    """
    img = img.convert("RGB") 
    
    img = img.resize((224, 224), Image.Resampling.BILINEAR)
    
    img_array = np.array(img).astype("float32") / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    return img_array

# --- MODULARISASI 2: Fungsi Utama Web Endpoint ---
def web_predict(img_path, model, class_names):
    """
    Fungsi backend API utama dengan perlindungan Memory Leak.
    """
    try:
        # 1. Verifikasi Keamanan File dengan Context Manager 
        with Image.open(img_path) as img_verify:
            img_verify.verify() 
        
        # 2. Buka gambar untuk diproses (Juga pakai Context Manager agar aman!)
        with Image.open(img_path) as img:
            img_batch = preprocess_image(img)
        
        # 3. Prediksi Model
        preds = model.predict(img_batch, verbose=0)
        class_idx = np.argmax(preds)
        
        # 4. Format Hasil
        result = {
            "status": "success",
            "diagnosis": class_names[class_idx],
            "confidence": float(np.max(preds) * 100) 
        }
        return result
        
    except UnidentifiedImageError:
        return {"status": "error", "message": "File yang diunggah rusak atau bukan gambar!"}
    except Exception as e:
        return {"status": "error", "message": f"Terjadi kesalahan sistem: {str(e)}"}

print("Sistem backend web siap digunakan")

Sistem backend web siap digunakan
